<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [ ]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122


In [ ]:
#DRIVE
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch

# Load the model
VERBOSE = True
CHOSEN = 'llama'
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"},
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf"},
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)

def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Summarize
1. Give the model a rulebook and prompt it to explain the game in simple, conversational terms to a child or other audiences. -> tree decomposition to test how well the model did
2. Test the ability of the model to find analogies of rules (?)
3. Test the ability to extract if-then rules (?)
4. Organize the rules of into a hierarchy: top-level objectives, mid-level phases, low-level actions. (?) (look into the paper)

In [ ]:
prompts = [("kid", """You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation"""),
           ("analogies", """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the rules to a child by comparing it to something they already know (e.g. some other famous board games).
           se the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n"""),
]
file_names = [ 'dominion','7_wonders', 'catan', 'power_grid_recharged','ticket_to_ride',]
iterations = 1

output_dict = {str(g):{pn: {str(it): '' for it in range(iterations) } for pn in prompts.keys()} for g in file_names}

for game,prompt_name,prompt,it in tqdm([(f,pn,p,it) for f in file_names for (pn,p) in prompts for it in range(iterations)]):
    rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
    out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)['choices'][0]['message']['content']
    if(VERBOSE):
        print(out)
    output_dict[game][prompt_name][str(it)] = out

with open(f'drive/MyDrive/NLP_proj/{CHOSEN}_summarize.out','w') as f:
    json.dump(dict(output_dict),f)

100%|██████████| 10/10 [03:58<00:00, 23.84s/it]


defaultdict(dict,
            {'llama-dominion-kid': {'0': "Hey there, young adventurer! Let's talk about the game Dominion. \n\n**What's the goal of the game?**\nThe goal of the game is to build a deck of cards that will help you win the most points. You'll be collecting cards that will give you money, help you buy new cards, and even defend against other players.\n\n**How do you win the game?**\nYou win the game by having the most points at the end. Points are earned by collecting Victory cards, which are worth points. The player with the most points wins the game!\n\n**What happens during a turn?**\nA turn has three phases: Action, Buy, and Clean-up.\n\n1. **Action phase**: You can play one Action card from your hand. Action cards do cool things like give you more cards, money, or even defend against other players.\n2. **Buy phase**: You can play Treasure cards to earn money, and then use that money to buy new cards from the Supply.\n3. **Clean-up phase**: You put all the cards you 

In [4]:
prompts = {"kid": """You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation""",
           "analogies": """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the rules to a child by comparing it to something they already know (e.g. some other famous board games).
           se the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n"""
           }

file_names = [ 'dominion','7_wonders', 'catan', 'power_grid_recharged','ticket_to_ride',]
iterations = 1
#output_dict = {str(g):{'lvl'+str(l): {str(it): '' for it in range(iterations)} for l in range(5)}      for g in file_names}
outputs =     {str(g):{pn:           {str(it): '' for it in range(iterations)} for pn in prompts.keys()} for g in file_names}
print(outputs)

{'dominion': {'kid': {'0': ''}, 'analogies': {'0': ''}}, '7_wonders': {'kid': {'0': ''}, 'analogies': {'0': ''}}, 'catan': {'kid': {'0': ''}, 'analogies': {'0': ''}}, 'power_grid_recharged': {'kid': {'0': ''}, 'analogies': {'0': ''}}, 'ticket_to_ride': {'kid': {'0': ''}, 'analogies': {'0': ''}}}
